# Notebook 2  --  Graph Theory & Spectral Convolution

This notebook builds the mathematical intuition behind Graph Convolutional
Networks (GCNs).  We go from the basic adjacency matrix all the way to
understanding *why* $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$
is the right normalisation.

---

## Contents

1. Graphs as matrices (adjacency, degree, Laplacian)
2. Graph signals and diffusion
3. The graph Laplacian and its eigendecomposition
4. Spectral graph convolution
5. The Kipf & Welling simplification
6. Intuition: why normalisation matters

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

## 1. Graphs as Matrices

A graph $G = (V, E)$ with $N$ nodes is represented by its
**adjacency matrix** $A \in \{0,1\}^{N \times N}$:

$$A_{ij} = 1 \text{ if } (i,j) \in E, \text{ else } 0$$

The **degree matrix** $D$ is diagonal: $D_{ii} = \sum_j A_{ij}$.

The **graph Laplacian** is: $L = D - A$

In [ ]:
# Small example: a path graph  0-1-2-3-4
N = 5
A = np.zeros((N, N))
edges = [(0,1),(1,2),(2,3),(3,4)]
for i,j in edges:
    A[i,j] = A[j,i] = 1.0

D = np.diag(A.sum(axis=1))
L = D - A

print("Adjacency matrix A:")
print(A.astype(int))
print("\nDegree matrix D (diagonal):")
print(D.astype(int))
print("\nLaplacian L = D - A:")
print(L.astype(int))

## 2. Graph Signals and Diffusion

A **graph signal** $\mathbf{x} \in \mathbb{R}^N$ assigns a scalar value
to each node (e.g., temperature, importance score).

Multiplying $A\mathbf{x}$ **aggregates** each node's neighbours:

$$(A\mathbf{x})_i = \sum_{j \in \mathcal{N}(i)} x_j$$

This is the fundamental message-passing operation!  GCNs repeat this
aggregation with learned linear transformations.

In [ ]:
# A signal with a spike at node 2
x = np.array([0., 0., 1., 0., 0.])

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
steps = [x]
for _ in range(3):
    steps.append(A @ steps[-1])

for t, (ax, sig) in enumerate(zip(axes, steps)):
    ax.bar(range(N), sig, color='steelblue')
    ax.set_title(f'$A^{t}\\mathbf{{x}}$ (step {t})')
    ax.set_xticks(range(N))
    ax.set_xlabel('Node')
    ax.set_ylim(-0.1, 2.5)
    ax.grid(alpha=0.3)

plt.suptitle('Diffusion of a spike signal across a path graph', y=1.02)
plt.tight_layout()
plt.show()
print("Notice: the spike spreads to neighbours with each multiplication.")

## 3. Spectral Analysis of the Laplacian

The normalised Laplacian is $\mathcal{L} = I - D^{-1/2}AD^{-1/2}$.

It is symmetric positive semi-definite, so it has a real eigendecomposition:

$$\mathcal{L} = U \Lambda U^\top$$

where $\Lambda = \text{diag}(\lambda_0 \le \lambda_1 \le \ldots \le \lambda_{N-1})$
and $U$ are orthonormal eigenvectors (**graph Fourier basis**).

**Key properties:**
- $\lambda_0 = 0$ always (constant signal is in the null space)
- $\lambda_\max \le 2$ for the normalised Laplacian
- Low eigenvalues -> smooth signals (slowly varying across the graph)
- High eigenvalues -> high-frequency signals (oscillate rapidly)

In [ ]:
# Normalised Laplacian
deg = A.sum(axis=1)
D_inv_sqrt = np.diag(np.where(deg > 0, deg**-0.5, 0))
L_norm = np.eye(N) - D_inv_sqrt @ A @ D_inv_sqrt

eigenvalues, eigenvectors = np.linalg.eigh(L_norm)

print("Eigenvalues of normalised Laplacian:")
for i, lam in enumerate(eigenvalues):
    print(f"  lambda_{i} = {lam:.4f}")

fig, axes = plt.subplots(1, N, figsize=(14, 3))
for i, ax in enumerate(axes):
    ax.bar(range(N), eigenvectors[:, i], color='coral')
    ax.set_title(f'$u_{i}$\n$\\lambda={eigenvalues[i]:.2f}$')
    ax.set_xticks(range(N))
    ax.axhline(0, color='k', linewidth=0.5)
    ax.grid(alpha=0.3)
plt.suptitle('Graph Fourier basis vectors (eigenvectors of L_norm)', y=1.02)
plt.tight_layout()
plt.show()

## 4. Spectral Graph Convolution

Classical convolution in Euclidean space is defined via the Fourier transform.
On graphs, we use the graph Fourier transform:

$$\hat{\mathbf{x}} = U^\top \mathbf{x}  \qquad (\text{graph Fourier transform})$$

A spectral filter $g_\theta$ is a diagonal matrix in frequency space:

$$g_\theta * \mathbf{x} = U\, g_\theta(\Lambda)\, U^\top \mathbf{x}$$

This is exact but expensive: $O(N^2)$ to store $U$ and $O(N^3)$ to compute.

### Chebyshev Approximation (ChebNet)

Hammond et al. (2011) proposed approximating $g_\theta(\Lambda)$ with a
degree-$K$ Chebyshev polynomial:

$$g_\theta(\Lambda) \approx \sum_{k=0}^{K} \theta_k T_k(\tilde{\Lambda})
\qquad \tilde{\Lambda} = \frac{2}{\lambda_{\max}}\Lambda - I$$

This brings complexity down to $O(K|E|)$  --  linear in the number of edges.

## 5. The Kipf & Welling (2017) Simplification

Setting $K=1$ and $\lambda_{\max} \approx 2$ (first-order approximation):

$$g_\theta * \mathbf{x} \approx \theta_0 \mathbf{x} + \theta_1 (\mathcal{L} - I)\mathbf{x}
= \theta_0 \mathbf{x} - \theta_1 D^{-1/2}AD^{-1/2}\mathbf{x}$$

Further constraining $\theta = \theta_0 = -\theta_1$ (single parameter per filter):

$$g_\theta * \mathbf{x} = \theta(I + D^{-1/2}AD^{-1/2})\mathbf{x}$$

The term $I + D^{-1/2}AD^{-1/2}$ has eigenvalues in $[0, 2]$, which can
cause numerical instability. The **renormalisation trick** adds self-loops first:

$$\tilde{A} = A + I, \quad \tilde{D}_{ii} = \sum_j \tilde{A}_{ij}$$

$$\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$$

This gives eigenvalues in $(0, 1]$  --  numerically stable.

In [ ]:
def compute_A_hat(A):
    """Compute the renormalised adjacency A_hat for a given A."""
    N = A.shape[0]
    A_tilde = A + np.eye(N)
    d_tilde = A_tilde.sum(axis=1)
    D_inv_sqrt = np.diag(d_tilde**-0.5)
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt

A_hat = compute_A_hat(A)
eigs_Ahat = np.linalg.eigvalsh(A_hat)

print(f"Eigenvalue range of A_hat: [{eigs_Ahat.min():.4f}, {eigs_Ahat.max():.4f}]")
print("All eigenvalues in (0, 1]:", np.all((eigs_Ahat > -1e-10) & (eigs_Ahat <= 1.0 + 1e-10)))

print("\nA_hat (renormalised adjacency):")
np.set_printoptions(precision=3, suppress=True)
print(A_hat)

## 6. Intuition: Why Normalisation Matters

Without normalisation, high-degree nodes accumulate large feature values
and low-degree nodes receive very little signal.  The symmetric normalisation
$\tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ weights each edge by the
geometric mean of the endpoint degrees:

$$\hat{A}_{ij} = \frac{1}{\sqrt{\tilde{d}_i \cdot \tilde{d}_j}}$$

A highly connected hub contributes less per edge  --  the signal is spread fairly.

In [ ]:
# Star graph: node 0 connects to all others (a hub)
N_star = 6
A_star = np.zeros((N_star, N_star))
for i in range(1, N_star):
    A_star[0, i] = A_star[i, 0] = 1.0

# Without normalisation: node 0 degree = 5
A_tilde_raw = A_star + np.eye(N_star)

A_hat_star = compute_A_hat(A_star)

x_star = np.ones(N_star)  # uniform signal

aggregated_raw  = A_tilde_raw @ x_star
aggregated_norm = A_hat_star  @ x_star

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(range(N_star), aggregated_raw, color='tomato')
axes[0].set_title('Without normalisation (A + I)')
axes[0].set_xlabel('Node index')
axes[0].set_ylabel('Aggregated value')
axes[0].set_xticks(range(N_star))
axes[0].set_xticklabels([f'Hub' if i==0 else f'{i}' for i in range(N_star)])
axes[0].grid(alpha=0.3)

axes[1].bar(range(N_star), aggregated_norm, color='steelblue')
axes[1].set_title('With normalisation (~)')
axes[1].set_xlabel('Node index')
axes[1].set_xticks(range(N_star))
axes[1].set_xticklabels([f'Hub' if i==0 else f'{i}' for i in range(N_star)])
axes[1].grid(alpha=0.3)

plt.suptitle('Star graph  --  effect of normalisation on aggregated signal', y=1.02)
plt.tight_layout()
plt.show()

print(f"Hub aggregated value (raw):  {aggregated_raw[0]:.2f}  (6-> the leaves)")
print(f"Hub aggregated value (norm): {aggregated_norm[0]:.2f}  (comparable to leaves)")

## 7. Layer-by-Layer Propagation on a Small Graph

Let's manually run two GCN layers on a toy graph to build concrete intuition
for what the network is computing.

In [ ]:
np.random.seed(0)

# Small graph: two communities
N_toy = 8
A_toy = np.zeros((N_toy, N_toy))
# Community 1: nodes 0-3
for i in range(4):
    for j in range(i+1, 4):
        A_toy[i,j] = A_toy[j,i] = 1.0
# Community 2: nodes 4-7
for i in range(4, 8):
    for j in range(i+1, 8):
        A_toy[i,j] = A_toy[j,i] = 1.0
# Bridge: node 3 - node 4
A_toy[3,4] = A_toy[4,3] = 1.0

A_hat_toy = compute_A_hat(A_toy)

# 2D features distinguishing the two communities
X_toy = np.zeros((N_toy, 2))
X_toy[:4, 0] = 1.0   # community 1 has feature 0
X_toy[4:, 1] = 1.0   # community 2 has feature 1

# Fixed weights (not trained  --  just to show propagation)
W1 = np.array([[1., 0.], [0., 1.]])  # identity
W2 = np.array([[1., -1.], [-1., 1.]])

H1 = np.maximum(0, A_hat_toy @ X_toy @ W1)
H2 = A_hat_toy @ H1 @ W2

print("Input features X:")
print(X_toy)
print("\nAfter layer 1 (H1):")
np.set_printoptions(precision=3, suppress=True)
print(H1)
print("\nAfter layer 2 (H2):")
print(H2)
print("\nNodes in community 1 (0-3) vs community 2 (4-7) become distinguishable.")

## Summary

| Concept | Formula | Role in GCN |
|---------|---------|-------------|
| Adjacency | $A_{ij} = 1$ if edge $(i,j)$ exists | Encodes graph structure |
| Degree matrix | $D_{ii} = \sum_j A_{ij}$ | Normalisation denominator |
| Laplacian | $L = D - A$ | Measures signal smoothness |
| Spectral conv | $U g(\Lambda) U^\top x$ | Principled graph convolution |
| Self-loops | $\tilde{A} = A + I$ | Node aggregates its own features |
| Renorm trick | $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$ | Stable eigenvalues in $(0,1]$ |
| GCN layer | $H' = \sigma(\hat{A}HW)$ | Learnable spectral filter |

The two-layer GCN with log-softmax and NLL loss is trained end-to-end by
gradient descent  --  the weights $W$ learn to amplify features that are
discriminative after two rounds of neighbourhood aggregation.